# Tutoriel 5

# Se familiariser avec les conditions aux bords

L'équation de diffusion décrit ce qui se passe **à l'intérieur** du domaine. Elle ne dit rien de ce qui se passe à ses deux extrémités : c'est le rôle des **conditions aux bords**. Ce sont elles qui décident si un mur se refroidit ou reste chaud, si un polluant s'échappe ou reste piégé.

Ce tutoriel ne construit pas de modèle : il montre uniquement **comment on implémente** chaque type de condition.

## 1. Pourquoi les bords sont à part

La discrétisation de la diffusion se fait en trois étapes :

```python
qx      = - D * ( T[1:] - T[:-1] ) / dx   # taille nx-1
dTdt    = - ( qx[1:] - qx[:-1] ) / dx     # taille nx-2
T[1:-1] += dTdt * dt                      # taille nx-2
```

**Regardez la troisième ligne : la mise à jour porte sur `T[1:-1]`, donc sur tous les points sauf le premier et le dernier.** Les deux points de bord ne sont **jamais** touchés par l'équation.

Conséquence : si l'on n'écrit aucune condition aux bords, `T[0]` et `T[-1]` gardent leur valeur initiale pendant toute la simulation. **Ne rien écrire, c'est donc quand même imposer une condition** — un Dirichlet figé, que l'on n'a pas choisi.

## 2. Les trois types de conditions

Pour le bord de gauche (`T[0]`) ; le bord de droite s'écrit symétriquement avec `T[-1]` et `T[-2]`.

| Type | Ce que l'on impose | Code |
|---|---|---|
| **Dirichlet** | la température vaut $a$ | `T[0] = a` |
| **Neumann** | le gradient vaut $a$ : $\frac{dT}{dx}(0) = a$ | `T[0] = T[1] - dx*a` |
| **Neumann, flux nul** | cas particulier $a = 0$ | `T[0] = T[1]` |

Le Neumann se retrouve en discrétisant la dérivée : $\frac{dT}{dx}(0) = a$ devient $\frac{T_1 - T_0}{dx} = a$, qui se réécrit $T_0 = T_1 - a\,dx$.

**Attention :** c'est bien `dx`, le pas d'**espace**, qui intervient — et non `dt`.

Voyons concrètement ce que chaque ligne fait au premier point d'un vecteur :

In [ ]:
import numpy as np

dx = 0.01                                    # pas d'espace, m
T  = np.array([35.0, 30.0, 25.0, 22.0, 20.0])   # une temperature quelconque
print("au depart                :", T)

T_dir = np.copy(T) ; T_dir[0] = 5.0          # Dirichlet : on impose la valeur
print("Dirichlet  T[0] = 5      :", T_dir)

a = -100.0                                   # gradient impose, degres/m
T_neu = np.copy(T) ; T_neu[0] = T_neu[1] - dx*a
print("Neumann    dT/dx = -100  :", T_neu)

T_nul = np.copy(T) ; T_nul[0] = T_nul[1]     # flux nul : le bord copie son voisin
print("flux nul   dT/dx = 0     :", T_nul)

Trois lignes, trois comportements. Le Dirichlet **écrase** la valeur du bord ; le Neumann la **calcule** à partir du voisin et du gradient voulu ; le flux nul **recopie** simplement le voisin, de sorte que la pente au bord soit nulle et que rien ne traverse.

## 3. Où les écrire dans le code

Les conditions aux bords agissent **à tout instant** : il faut donc les ré-appliquer **à chaque itération**, juste après la mise à jour de l'équation.

```python
for it in range(nt):

    # 1) equation de diffusion, a l'interieur du domaine
    qx      = - D * ( T[1:] - T[:-1] ) / dx
    T[1:-1] += - dt * ( qx[1:] - qx[:-1] ) / dx

    # 2) conditions aux bords, sur les deux points restants
    T[0]  = T_gauche        # Dirichlet a gauche
    T[-1] = T[-2]           # flux nul a droite
```

Rien n'oblige à mettre le même type des deux côtés : ce sont les conditions **mixtes**, et c'est le cas le plus fréquent en pratique.

## 4. Ce que chaque choix produit

Sur un mur dont la moitié gauche est chaude et la moitié droite froide, les quatre combinaisons donnent des solutions **complètement différentes** :

| Conditions | Ce qui se passe |
|---|---|
| Dirichlet / Dirichlet | le profil tend vers une **droite** entre les deux températures imposées |
| flux nul / flux nul | domaine fermé : la marche s'aplatit vers la **moyenne**, et la chaleur totale est **conservée** |
| flux imposé / flux nul | on injecte en permanence sans rien laisser sortir : la température **monte sans se stabiliser** |
| Dirichlet / flux nul | tout le mur finit à la température **imposée à gauche** |

D'où un contrôle très utile : surveiller la quantité totale `np.sum(T)*dx`. Si votre domaine est censé être fermé et que ce nombre dérive au fil des itérations, vos conditions aux bords sont fausses.

## À expérimenter

1. Dans la cellule ci-dessus, changez le gradient `a` de `-100` à `+100`. Le bord devient-il plus chaud ou plus froid que son voisin ? Dans quel sens la chaleur entre-t-elle ?
2. Que donnerait la condition de flux nul au bord **droit** ? Écrivez-la et vérifiez sur le vecteur.